# Phase 4: SQL Analytics

## Objective
Simulate a real business database environment by loading cleaned data into SQLite and answering key business questions using SQL.

## Database Schema
- **customers** — Customer information and location
- **orders** — Order status and timestamps  
- **order_items** — Products within each order and pricing
- **products** — Product details and categories
- **payments** — Payment methods and values
- **reviews** — Customer ratings and feedback
- **sellers** — Seller information and location

## Tools Used
- SQLite (via Python)
- Pandas for displaying results

In [5]:
import sqlite3
import pandas as pd

# Force close any existing connections
try:
    conn.close()
except:
    pass

# Delete and recreate database
import os
db_path = '../../02_Data/olist.db'
if os.path.exists(db_path):
    os.remove(db_path)

# Fresh connection
conn = sqlite3.connect(db_path)

# Load all cleaned CSVs
customers = pd.read_csv('../../02_Data/02_Cleaned_Data/customers_clean.csv')
orders = pd.read_csv('../../02_Data/02_Cleaned_Data/orders_clean.csv')
order_items = pd.read_csv('../../02_Data/02_Cleaned_Data/order_items_clean.csv')
products = pd.read_csv('../../02_Data/02_Cleaned_Data/products_clean.csv')
payments = pd.read_csv('../../02_Data/02_Cleaned_Data/payments_clean.csv')
reviews = pd.read_csv('../../02_Data/02_Cleaned_Data/reviews_clean.csv')
sellers = pd.read_csv('../../02_Data/02_Cleaned_Data/sellers_clean.csv')

# Save to database
customers.to_sql('customers', conn, if_exists='replace', index=False)
orders.to_sql('orders', conn, if_exists='replace', index=False)
order_items.to_sql('order_items', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
payments.to_sql('payments', conn, if_exists='replace', index=False)
reviews.to_sql('reviews', conn, if_exists='replace', index=False)
sellers.to_sql('sellers', conn, if_exists='replace', index=False)

print("✅ Database created successfully!")
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))

✅ Database created successfully!
          name
0    customers
1       orders
2  order_items
3     products
4     payments
5      reviews
6      sellers


In [4]:
import pandas as pd
import sqlite3

# Reconnect to database
conn = sqlite3.connect('../../02_Data/olist.db')

print("✅ Ready!")

✅ Ready!


### ✅ Database Setup Complete
All 7 tables successfully loaded into SQLite database.
Total tables: customers, orders, order_items, products, payments, reviews, sellers

## 🔍 Query 1: Business Snapshot

*The first thing any manager asks — how big is this business?*

Before any deep dive, I wanted to understand the scale. Total revenue, orders placed, unique customers served, and the average spend per order — all filtered for delivered orders only, since cancelled orders don't represent real business.

---

In [6]:
# Query 1: Total Revenue, Orders and Customers - Business Overview
query = """
SELECT 
    COUNT(DISTINCT o.order_id) as total_orders,
    COUNT(DISTINCT o.customer_id) as total_customers,
    ROUND(SUM(p.payment_value), 2) as total_revenue,
    ROUND(AVG(p.payment_value), 2) as avg_order_value
FROM orders o
JOIN payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
"""

result = pd.read_sql(query, conn)
result

,total_orders,total_customers,total_revenue,avg_order_value
0,96477,96477,15422461.77,153.07


**What this tells us:**

Olist processed over 96,000 delivered orders generating **$15.4 million in revenue**. The average customer spends around **$153 per order** — which is a solid number for an e-commerce marketplace.

One interesting observation — total orders and total customers are exactly the same, which tells us that **almost every customer bought only once**. This is a massive retention problem and something we'll dig deeper into in the Customer Intelligence phase.

---

## 📈 Query 2: Monthly Revenue Trend
*Total revenue is one number. The trend is the real story.*

I wanted to see how sales moved month by month — to find growth patterns, seasonal spikes, and any red flags worth investigating.

---

In [5]:
query = """
SELECT 
    strftime('%Y-%m', o.order_purchase_timestamp) AS month,
    COUNT(DISTINCT o.order_id)                     AS total_orders,
    ROUND(SUM(p.payment_value), 2)                 AS monthly_revenue
FROM orders o
JOIN payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY month
ORDER BY month
"""
pd.read_sql(query, conn)

,month,total_orders,monthly_revenue
0,2016-10,265,46566.71
1,2016-12,1,19.62
2,2017-01,750,127545.67
3,2017-02,1653,271298.65
4,2017-03,2546,414369.39
5,2017-04,2303,390952.18
6,2017-05,3546,567066.73
7,2017-06,3135,490225.60
8,2017-07,3872,566403.93
9,2017-08,4193,646000.61


Revenue grew steadily through 2017 peaking in November — Black Friday. This tells the business exactly when to run promotions and when to stock up inventory.

---

## 🗺️ Query 3: Top Revenue States
*Brazil has 26 states — but not all of them pull their weight.*

I mapped revenue by state to find where Olist's strongest markets are, and where the growth opportunity lies.

---

In [6]:
query = """
SELECT 
    c.customer_state               AS state,
    COUNT(DISTINCT o.order_id)     AS total_orders,
    ROUND(SUM(p.payment_value), 2) AS total_revenue
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN payments p  ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY state
ORDER BY total_revenue DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,state,total_orders,total_revenue
0,SP,40500,5770266.19
1,RJ,12350,2055690.45
2,MG,11354,1819277.61
3,RS,5345,861802.40
4,PR,4923,781919.55
5,SC,3546,595208.40
6,BA,3256,591270.60
7,DF,2080,346146.17
8,GO,1957,334294.22
9,ES,1995,317682.65


💡 São Paulo dominates by a massive margin. For Olist, SP isn't just the biggest market — it's practically the whole business. Expanding into other states should be a top strategic priority.

---

##  Query 4: Best Performing Product Categories
*What are people actually buying on Olist?*

Knowing which categories drive the most revenue helps the business decide where to invest — more sellers, better logistics, or targeted campaigns.

---

In [7]:
query = """
SELECT 
    p.product_category_name        AS category,
    COUNT(DISTINCT oi.order_id)    AS total_orders,
    ROUND(SUM(oi.price), 2)        AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
WHERE p.product_category_name IS NOT NULL
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,category,total_orders,total_revenue
0,beleza_saude,8836,1258681.34
1,relogios_presentes,5624,1205005.68
2,cama_mesa_banho,9417,1036988.68
3,esporte_lazer,7720,988048.97
4,informatica_acessorios,6689,911954.32
5,moveis_decoracao,6449,729762.49
6,cool_stuff,3632,635290.85
7,utilidades_domesticas,5884,632248.66
8,automotivo,3897,592720.11
9,ferramentas_jardim,3518,485256.46


Top categories reveal considered purchases not impulse buys — high revenue with moderate order counts means people spend more per item. These categories need different marketing than low-ticket products.

---

## 👑 Query 5: Top Customers by Revenue
*Who are the customers Olist absolutely cannot afford to lose?*

In any business a small group of power buyers drives a disproportionate share of revenue. Knowing who they are is step one to keeping them.

---

In [8]:
query = """
SELECT 
    o.customer_id,
    COUNT(DISTINCT o.order_id)      AS total_orders,
    ROUND(SUM(p.payment_value), 2)  AS total_spent
FROM orders o
JOIN payments p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
GROUP BY o.customer_id
ORDER BY total_spent DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,customer_id,total_orders,total_spent
0,1617b1357756262bfa56ab541c47bc16,1,13664.08
1,ec5b2ba62e574342386871631fafd3fc,1,7274.88
2,c6e2731c5b391845f6800c97401a43a9,1,6929.31
3,f48d464a0baaea338cb25f816991ab1f,1,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,1,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,1,6081.54
6,df55c14d1476a9a3467f131269c2477f,1,4950.34
7,24bbf5fd2f2e1b359ee7de94defc4a15,1,4764.34
8,3d979689f636322c62418b6346b1c6d2,1,4681.78
9,1afc82cd60e303ef09b4ef9837c9505c,1,4513.32


💡 Top customers have spent significantly more than the $153 platform average. These rare repeat buyers are exactly the profile Olist should be trying to create more of through loyalty incentives.

---

## 🚚 Query 6: Delivery Performance
*Is Olist actually keeping its delivery promises?*

The customer experience doesn't end at checkout — it ends when the package arrives. Late deliveries are the top reason customers leave bad reviews and never come back.

---

In [9]:
query = """
SELECT
    COUNT(*) AS total_delivered,
    SUM(CASE WHEN order_delivered_customer_date <= order_estimated_delivery_date 
        THEN 1 ELSE 0 END) AS on_time,
    SUM(CASE WHEN order_delivered_customer_date > order_estimated_delivery_date 
        THEN 1 ELSE 0 END) AS late,
    ROUND(AVG(julianday(order_delivered_customer_date) - 
        julianday(order_purchase_timestamp)), 1) AS avg_delivery_days
FROM orders
WHERE order_status = 'delivered'
AND order_delivered_customer_date IS NOT NULL
"""
pd.read_sql(query, conn)

,total_delivered,on_time,late,avg_delivery_days
0,96470,88644,7826,12.6


💡 Most orders arrive on time — but late deliveries when they happen tend to be significantly late, not just by a day. That long tail of very delayed orders is what drags down review scores in specific regions.

---

## ⭐ Query 7: Review Scores by Category
*Which categories delight customers — and which frustrate them?*

Review scores are the customer's voice. Mapping them to categories pinpoints exactly where quality issues are concentrated — and gives the business a clear target to fix.

> 💡 This query uses a **CTE** to organize logic cleanly and **RANK()** window function to rank categories — two techniques standard in professional SQL work.

---

In [10]:
query = """
WITH category_reviews AS (
    SELECT 
        p.product_category_name       AS category,
        ROUND(AVG(r.review_score), 2) AS avg_rating,
        COUNT(r.review_id)            AS total_reviews
    FROM reviews r
    JOIN orders o       ON r.order_id = o.order_id
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p     ON oi.product_id = p.product_id
    WHERE p.product_category_name IS NOT NULL
    GROUP BY category
)
SELECT *,
    RANK() OVER (ORDER BY avg_rating DESC) AS rating_rank
FROM category_reviews
WHERE total_reviews > 100
ORDER BY avg_rating DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,category,avg_rating,total_reviews,rating_rank
0,livros_interesse_geral,4.45,549,1
1,livros_tecnicos,4.37,266,2
2,alimentos_bebidas,4.32,279,3
3,malas_acessorios,4.32,1088,3
4,fashion_calcados,4.23,261,5
5,alimentos,4.22,495,6
6,papelaria,4.19,2507,7
7,pet_shop,4.19,1939,7
8,eletrodomesticos,4.17,806,9
9,pcs,4.17,200,9


💡 High-rated categories are where Olist's seller quality is strongest. Low-rated ones are where complaints concentrate — and where seller audits or stricter quality controls would make the biggest difference.

---

## 💳 Query 8: Revenue by Payment Method
*How do customers prefer to pay — and what does that tell us?*

Payment behavior reveals customer demographics and financial confidence. It also tells us where the business might be vulnerable if one payment method is disrupted.

---

In [11]:
query = """
SELECT 
    payment_type,
    COUNT(*)                        AS total_transactions,
    ROUND(SUM(payment_value), 2)    AS total_revenue,
    ROUND(AVG(payment_value), 2)    AS avg_transaction_value
FROM payments
GROUP BY payment_type
ORDER BY total_revenue DESC
"""
pd.read_sql(query, conn)

,payment_type,total_transactions,total_revenue,avg_transaction_value
0,credit_card,76795,12542084.19,163.32
1,boleto,19784,2869361.27,145.03
2,voucher,5775,379436.87,65.70
3,debit_card,1529,217989.79,142.57
4,not_defined,3,0.00,0.00


💡 Credit card is dominant — expected. But the high use of installments tells us Olist's customers are price-sensitive and rely on splitting payments. Any price increase strategy needs to account for this behavior.

---